# Website Summarizer with OpenAI

This notebook fetches text from a webpage, cleans it, and generates a summary using the OpenAI chat Completions API.

## Goals 
- Learn how to retrieve website content
- Clean HTML into plain text
- Send content to an LLM for summarization
- Understand the limits of direct summrization


In [ ]:
%pip install openai python-dotenv requests beautifulsoup4

In [ ]:
import os
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
## Load environment variables
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("No OpenAI API key found in environment variables")

client = OpenAI(api_key=api_key)
print("OpenAI client initialized successfully")

In [ ]:
## Fetch the webpage content
import requests
## Define the URL to scrape
url = "https://en.wikipedia.org/wiki/Nepal"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers, timeout=10)
response.raise_for_status()

html = response.text
print(f"Fetched {len(html)} characters")

In [ ]:
from bs4 import BeautifulSoup
import re

soup = BeautifulSoup(html, "html.parser")

# 1. Remove unwanted tags
for tag in soup(["script", "style", "noscript", "template", "iframe", "sup"]):
    tag.decompose()

# 2. Target main content (Wikipedia specific)
main_content = soup.find("div", {"id": "mw-content-text"})

if not main_content:
    raise ValueError("Main content not found")

# 3. Extract only meaningful text blocks
paragraphs = main_content.find_all(["p", "h2", "h3"])

clean_text = []

for p in paragraphs:
    text = p.get_text(" ", strip=True)
    
    # Remove citation markers like [1], [23]
    text = re.sub(r"\[\d+\]", "", text)
    
    # Skip empty or very short lines
    if len(text) > 50:
        clean_text.append(text)

# 4. Join into final text
final_text = "\n\n".join(clean_text)

print(final_text[:5000])  # limit output

In [ ]:
## Create a summary prompt
prompt = f"""
Summarize the following webpage content clearly and concisely.

Focus on:
- the main topic
- key points
- any important conclusions or takeaways

Webpage content:
{final_text}
"""



In [ ]:
## Call openAI
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant that summarizes text."},
        {"role": "user", "content": prompt}
    ],
    temperature=0.3,
)
summary = response.choices[0].message.content
print(summary)

In [ ]:
## Display the summary
display(Markdown(" ## Summary"))
display(Markdown(summary))

In [ ]:
## Show token usage
if response.usage:
    print(f"Tokens used: {response.usage.total_tokens}")
    print(f"Prompt tokens: {response.usage.prompt_tokens}")
    print(f"Completion tokens: {response.usage.completion_tokens}")
    print(f"Total cost: ${response.usage.total_tokens / 1000000 * 0.15:.6f}")
    
